# PageZero GRPO Training (Colab) - Unsloth + Local or Deployed BackendThis notebook trains the PageZero agent with GRPO using your existing training pipeline.You can run with any backend URL mode:- local_same_machine: use local Docker backend on the same machine as notebook- local_via_tunnel: use local Docker backend exposed by HTTPS tunnel (best for Colab)- deployed_space: use deployed backend URL (HF Space or VM URL)Only one variable controls this switch: ENV_URL (derived from BACKEND_MODE).

## 1) Install DependenciesIncludes Unsloth, TRL (>=0.29.0), vLLM, and OpenEnv runtime.

In [ ]:
import subprocessimport sysfrom importlib import metadatasubprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)core_packages = [    'transformers',    'datasets',    'accelerate',    'matplotlib',    'pandas',    'peft',    'trl>=0.29.0',    'huggingface_hub>=0.24.0,<1.0',    'openenv-core[core]>=0.2.1',]subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *core_packages], check=True)optional_packages = ['unsloth', 'vllm>=0.12.0']for pkg in optional_packages:    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg], check=False)    if result.returncode == 0:        print(f'Installed optional package: {pkg}')    else:        print(f'Skipping optional package on this runtime: {pkg}')for pkg in ['transformers', 'datasets', 'peft', 'trl', 'huggingface-hub']:    try:        print(f'{pkg}: {metadata.version(pkg)}')    except Exception:        print(f'{pkg}: NOT INSTALLED')

In [ ]:
import osprint('Restarting kernel to apply updated TRL version...')os._exit(0)

### Runtime RestartedAfter the cell above runs, the runtime will restart automatically. Once connected again, continue with Section 2.

## 2) ConfigurationSet token, backend mode, URLs, model, and training hyperparameters.Use this backend mode selector:- local_same_machine: local Docker backend on the same machine as notebook- local_via_tunnel: local Docker backend exposed by HTTPS tunnel (for Colab)- deployed_space: deployed backend URL (HF Space or VM endpoint)- custom_url: read URL from PAGEZERO_ENV_URL environment variable

In [ ]:
import osIN_COLAB = 'COLAB_GPU' in os.environ# --- HuggingFace token (optional unless pushing to hub) ---try:    from google.colab import userdata    if 'HF_TOKEN' not in os.environ:        token = userdata.get('HF_TOKEN')        if token:            os.environ['HF_TOKEN'] = tokenexcept Exception:    passif 'HF_TOKEN' not in os.environ:    print('WARNING: HF_TOKEN not found. Needed only for push_to_hub.')# --- Backend mode selector ---BACKEND_MODE = 'deployed_space'  # local_same_machine | local_via_tunnel | deployed_space | custom_urlLOCAL_SAME_MACHINE_URL = 'http://localhost:8000'LOCAL_TUNNEL_URL = 'https://YOUR-TUNNEL-URL.trycloudflare.com'DEPLOYED_SPACE_URL = 'https://pranayy-pagezero-agent.hf.space/'CUSTOM_ENV_URL = os.environ.get('PAGEZERO_ENV_URL', '').strip()if BACKEND_MODE == 'local_same_machine':    ENV_URL = LOCAL_SAME_MACHINE_URLelif BACKEND_MODE == 'local_via_tunnel':    ENV_URL = LOCAL_TUNNEL_URLelif BACKEND_MODE == 'deployed_space':    ENV_URL = DEPLOYED_SPACE_URLelif BACKEND_MODE == 'custom_url':    if not CUSTOM_ENV_URL:        raise ValueError("BACKEND_MODE='custom_url' but PAGEZERO_ENV_URL is empty.")    ENV_URL = CUSTOM_ENV_URLelse:    raise ValueError(f'Unknown BACKEND_MODE: {BACKEND_MODE}')if IN_COLAB and BACKEND_MODE == 'local_same_machine':    print('WARNING: localhost points to the Colab VM, not your laptop.')    print('Use local_via_tunnel or deployed_space for Colab.')# --- Repo / checkout config ---REPO_URL = 'https://github.com/Dishant-garg/PageZero-RL.git'USE_EXISTING_CHECKOUT = not IN_COLABEXISTING_CHECKOUT_DIR = os.getcwd() if not IN_COLAB else '/content/PageZero'CLONE_DIR = '/content/PageZero' if IN_COLAB else EXISTING_CHECKOUT_DIR# --- Model config ---USE_UNSLOTH_MODEL = False  # True will try 4-bit Unsloth model idUNSLOTH_MODEL_ID = 'unsloth/Qwen3-0.6B-bnb-4bit'BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'MODEL_ID = UNSLOTH_MODEL_ID if USE_UNSLOTH_MODEL else BASE_MODEL_IDFALLBACK_MODEL_ID = BASE_MODEL_ID# --- Training hyperparameters ---NUM_EPISODES = 50NUM_GENERATIONS = 8MAX_TURNS = 15MAX_NEW_TOKENS = 512VLLM_MODE = 'colocate'OUTPUT_DIR_BASE = 'outputs/pagezero-colab'# --- Hub push config ---PUSH_TO_HUB = FalseHUB_REPO = 'pranayy/pagezero_agent'print(f'IN_COLAB      : {IN_COLAB}')print(f'BACKEND_MODE  : {BACKEND_MODE}')print(f'ENV_URL       : {ENV_URL}')print(f'MODEL_ID      : {MODEL_ID}')print(f'EPISODES      : {NUM_EPISODES}')print(f'GENERATIONS   : {NUM_GENERATIONS}')

## 3) Project SetupIf running in Colab, clone your repository first.
If running from an existing local checkout, keep USE_EXISTING_CHECKOUT = True.

In [ ]:
import osimport subprocessif USE_EXISTING_CHECKOUT:    CLONE_DIR = EXISTING_CHECKOUT_DIR    if not os.path.exists(CLONE_DIR):        raise FileNotFoundError(f'Existing checkout not found: {CLONE_DIR}')else:    if 'YOUR_ORG' in REPO_URL:        raise ValueError('Please set REPO_URL to your real project repository URL first.')    if not os.path.exists(CLONE_DIR):        subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)os.chdir(CLONE_DIR)subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)print(f'Working directory: {os.getcwd()}')

## 4) Smoke Test - Verify Backend ConnectivityConnect to the selected backend URL, run reset, and execute one safe action.

In [ ]:
pwd()

In [ ]:
ls

In [ ]:
import requestsimport osimport sysif os.path.abspath('..') not in sys.path:    sys.path.append(os.path.abspath('..'))from PageZero.client import PageZeroEnvClientfrom PageZero.models import PageZeroActionprint(f'Testing backend: {ENV_URL}')for suffix in ['/health']:    url = ENV_URL.rstrip('/') + suffix    try:        r = requests.get(url, timeout=10)        print(f'{url} -> {r.status_code}')    except Exception as e:        print(f'{url} -> request failed: {e}')env = PageZeroEnvClient(base_url=ENV_URL)try:    reset_result = env.reset()    obs = reset_result.observation    print('Reset successful')    print(f'Step {obs.step}/{obs.max_steps}')    print(f'Active alerts: {obs.active_alerts}')    step_result = env.step(PageZeroAction(tool='check_alerts', args={}))    step_obs = step_result.observation    print(f'Step reward: {step_result.reward}')    print(f'Done: {step_result.done}')    print(f'SLA status: {step_obs.sla_status}')finally:    env.close()

## 5) Import Training Utilities

In [ ]:
import csvimport loggingimport osimport subprocessimport sysfrom datetime import datetimefrom pathlib import Pathfrom importlib import metadatarequired_specs = {    'datasets': 'datasets',    'transformers': 'transformers',    'trl': 'trl>=0.29.0',}missing_specs = []for dist_name, pip_spec in required_specs.items():    try:        metadata.version(dist_name)    except metadata.PackageNotFoundError:        missing_specs.append(pip_spec)try:    hub_version = metadata.version('huggingface-hub')except metadata.PackageNotFoundError:    hub_version = Noneneeds_hub_fix = hub_version is None or int(hub_version.split('.')[0]) >= 1if needs_hub_fix or missing_specs:    install_specs = []    if needs_hub_fix:        install_specs.append('huggingface_hub>=0.24.0,<1.0')    install_specs.extend(missing_specs)    print('Installing/adjusting:', ', '.join(install_specs))    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *install_specs], check=True)print(f"huggingface-hub version: {metadata.version('huggingface-hub')}")for mod_name in list(sys.modules):    if mod_name.startswith(('transformers', 'peft', 'accelerate', 'trl')):        sys.modules.pop(mod_name, None)from datasets import Datasetfrom transformers import AutoTokenizerfrom trl import GRPOConfig, GRPOTrainerfrom peft import LoraConfigif os.path.abspath('..') not in sys.path:    sys.path.append(os.path.abspath('..'))from PageZero.train import (    SYSTEM_PROMPT,    PageZeroToolEnv,    plot_rewards,    patch_trl_vllm_compat,)patch_trl_vllm_compat()logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')logger = logging.getLogger(__name__)print('Imported training utilities successfully.')print('System prompt preview:')print(SYSTEM_PROMPT[:220] + '...')

## 6) Build GRPO Trainer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenPageZeroToolEnv.BASE_URL = ENV_URLprompt_messages = [    {'role': 'system', 'content': SYSTEM_PROMPT},    {'role': 'user', 'content': 'Diagnose and fix this production incident.'},]dataset = Dataset.from_dict({'prompt': [prompt_messages for _ in range(NUM_EPISODES)]})timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')output_dir = Path(f'{OUTPUT_DIR_BASE}-{timestamp}')output_dir.mkdir(parents=True, exist_ok=True)reward_log_path = output_dir / 'reward_log.csv'with open(reward_log_path, 'w', newline='') as f:    csv.writer(f).writerow(['episode', 'total_reward', 'diagnosis_reward', 'fix_reward', 'timestamp'])def log_episode(total_r, diag_r, fix_r):    # simple CSV logger for episodes    with open(reward_log_path, 'a', newline='') as f:        csv.writer(f).writerow([total_r, diag_r, fix_r, datetime.now().isoformat()])def reward_total(completions=None, environments=None, **kwargs):    if not environments:        count = len(completions) if completions is not None else 0        return [0.0 for _ in range(count)]    values = []    for env in environments:        total = float(getattr(env, 'total_reward', 0.0))        diag = float(getattr(env, 'diagnosis_reward', 0.0))        fix = float(getattr(env, 'fix_reward', 0.0))        values.append(total)        if not getattr(env, '_episode_logged', False):            log_episode(total, diag, fix)            env._episode_logged = True    return valuesdef reward_diagnosis(completions=None, environments=None, **kwargs):    if not environments:        count = len(completions) if completions is not None else 0        return [0.0 for _ in range(count)]    return [float(getattr(env, 'diagnosis_reward', 0.0)) for env in environments]def reward_fix(completions=None, environments=None, **kwargs):    if not environments:        count = len(completions) if completions is not None else 0        return [0.0 for _ in range(count)]    return [float(getattr(env, 'fix_reward', 0.0)) for env in environments]grpo_config = GRPOConfig(    use_vllm=True,    vllm_mode=VLLM_MODE,    output_dir=str(output_dir),    num_train_epochs=1,    learning_rate=2e-6,    lr_scheduler_type='cosine',    warmup_steps=2,    max_grad_norm=1.0,    gradient_accumulation_steps=8,    per_device_train_batch_size=1,    generation_batch_size=NUM_GENERATIONS,    num_generations=NUM_GENERATIONS,    max_completion_length=MAX_NEW_TOKENS,    max_tool_calling_iterations=MAX_TURNS,    logging_steps=1,    save_strategy='steps',    save_steps=10,    temperature=1.0,    report_to='none',    gradient_checkpointing=True,    gradient_checkpointing_kwargs={'use_reentrant': False},    save_total_limit=3,    loss_type='dapo',    mask_truncated_completions=True,    beta=0.01,)peft_config = LoraConfig(    r=16,    lora_alpha=32,    lora_dropout=0.05,    bias='none',    task_type='CAUSAL_LM',    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],)trainer = GRPOTrainer(    model=MODEL_ID,    processing_class=tokenizer,    reward_funcs=[reward_total, reward_diagnosis, reward_fix],    train_dataset=dataset,    args=grpo_config,    peft_config=peft_config,    environment_factory=PageZeroToolEnv,)print(f'Trainer initialized. Output path: {output_dir}')

## 7) Train, Plot Rewards, and Optional Hub Push

In [ ]:
import pandas as pdfrom IPython.display import display, Imageprint('Starting GRPO training...')print(f'  Model       : {MODEL_ID}')print(f'  Episodes    : {NUM_EPISODES}')print(f'  Generations : {NUM_GENERATIONS}')print(f'  Environment : {ENV_URL}')print()try:    trainer.train()finally:    PageZeroToolEnv.close_all()    try:        plot_rewards(reward_log_path, output_dir / 'reward_plot.png')    except Exception as e:        print(f'WARNING: could not generate reward plot: {e}')trainer.save_model(str(output_dir))print(f'Model saved to {output_dir}')if reward_log_path.exists():    df = pd.read_csv(reward_log_path)    display(df.tail(10))reward_png = output_dir / 'reward_plot.png'if reward_png.exists():    display(Image(filename=str(reward_png)))if PUSH_TO_HUB:    if 'HF_TOKEN' not in os.environ:        raise RuntimeError('HF_TOKEN not set. Add it in Colab secrets or env vars.')    trainer.push_to_hub(repo_id=HUB_REPO)    print(f'Model pushed to https://huggingface.co/{HUB_REPO}')else:    print('Skipping push. Set PUSH_TO_HUB=True to enable.')